# Geração do Dataset

Este notebook tem como objetivo, baixar o dataset bruto no kaggle remover colunas que podem deixar o modelo ou tendecioso ou que não impacta no objetivo. Além disso, fazer a divisão em train / val / test, com Entrada e Saída.

Vale ressaltar que o dataset tratado será disponível no Kaggle e nesse repositório está no gitignore

In [ ]:
# Baixar Dataset via API do Kaggle
import kagglehub

# Download da versão mais recente
path = kagglehub.dataset_download("brunommm/dados-climticos-inmet-2026-tratados")

print("Path to dataset files:", path)

C:\Users\caiof\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 78.4M/78.4M [00:05<00:00, 14.1MB/s]

Extracting files...


Path to dataset files: C:\Users\caiof\.cache\kagglehub\datasets\brunommm\dados-climticos-inmet-2026-tratados\versions\1


## Análise do Dataset
Agora preciso entender como funciona o dataset para realizar o seu treinamento

In [11]:
import pandas as pd
df = pd.read_csv("Conjunto_De_Dados/brunommm/dados-climticos-inmet-2026-tratados/versions/1/total_uf.csv", sep = ',', decimal = '.')

df.head()

,regiao,uf,estacao,latitude,longitude,altitude,timestamp_utc,data_br,hora_utc,mes,...,umidade_max_porcento,umidade_min_porcento,pressao_estacao_mb,pressao_max_mb,pressao_min_mb,precipitacao_total_mm,radiacao_global_kj_m2,vento_direcao_graus,vento_velocidade_ms,vento_rajada_ms
0,S,RS,VIAMAO,-30.115278,-50.929167,65.0,2026-03-17 00:00:00,17/03/2026,0,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,S,RS,VIAMAO,-30.115278,-50.929167,65.0,2026-03-17 01:00:00,17/03/2026,100,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,S,RS,VIAMAO,-30.115278,-50.929167,65.0,2026-03-17 02:00:00,17/03/2026,200,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,S,RS,VIAMAO,-30.115278,-50.929167,65.0,2026-03-17 03:00:00,17/03/2026,300,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,S,RS,VIAMAO,-30.115278,-50.929167,65.0,2026-03-17 04:00:00,17/03/2026,400,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Conclusão
- Para nosso objetivo, iremos eliminar alguns dados como:
    1. Colunas referente a localização: Nosso modelo não pode ficar tendecioso para uma perspectiva;
    2. Altitude: Não interfere na condição climática;
    3. Radiação Global e direção de vento não seria aplicado para a parte de IoT.

- Pensando nisso haverá uma divisão de lotes contendo:
    1. Data e horário;
    2. Umidade;
    3. Pressão;
    4. Precipitação;
    5. Mínimos e Máximos dos itens 2 ao 4.


## Execução
- Para isso armazenaremos em um JSON todos os dados referentes ao dia n e chamaremos de entrada
- Depois pegaremos o dia n+1 e faremos a mesma coisa, porém chamares de label;


In [ ]:
#Definimos a lista final contendo apenas os dados climáticos essenciais
colunas_manter = [
    # Data e horário
    'data_br', 'hora_utc', 'mes',
    # Umidade (Máxima e Mínima)
    'umidade_max_porcento', 'umidade_min_porcento',
    # Pressão (Geral, Máxima e Mínima)
    'pressao_estacao_mb', 'pressao_max_mb', 'pressao_min_mb',
    # Precipitação
    'precipitacao_total_mm'
]

#Filtramos o DataFrame
df_iot = df[colunas_manter]

df_iot.head()


,data_br,hora_utc,mes,umidade_max_porcento,umidade_min_porcento,pressao_estacao_mb,pressao_max_mb,pressao_min_mb,precipitacao_total_mm,vento_velocidade_ms,vento_rajada_ms
0,17/03/2026,0,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,17/03/2026,100,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,17/03/2026,200,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,17/03/2026,300,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,17/03/2026,400,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
import json
import random
from pathlib import Path

#Converter para a data para o pandas entender a ordem
df_iot['data_br'] = pd.to_datetime(df_iot['data_br'], format='%d/%m/%Y')

#Lote diário
df_diario = df_iot.groupby('data_br').agg({
    'mes': 'first',                     # Pega o mês daquele dia
    'umidade_max_porcento': 'max',      # A maior umidade registrada nas 24h
    'umidade_min_porcento': 'min',      # A menor umidade registrada nas 24h
    'pressao_estacao_mb': 'mean',       # A pressão média do dia
    'pressao_max_mb': 'max',            # A maior pressão registrada nas 24h
    'pressao_min_mb': 'min',            # A menor pressão registrada nas 24h
    'precipitacao_total_mm': 'sum'      # Soma toda a chuva que caiu nas 24h
}).sort_index().reset_index()

# Criar os pares entrada -> label
lotes_dataset = []

for i in range(len(df_diario) - 1):
    dia_n = df_diario.iloc[i].to_dict()
    dia_n_mais_1 = df_diario.iloc[i + 1].to_dict()

    # Converter Timestamp para string no formato brasileiro
    dia_n['data_br'] = dia_n['data_br'].strftime('%d/%m/%Y')
    dia_n_mais_1['data_br'] = dia_n_mais_1['data_br'].strftime('%d/%m/%Y')

    lote = {
        'entrada': dia_n,
        'label': dia_n_mais_1
    }

    lotes_dataset.append(lote)

# Embaralhar os pares completos
random.seed(42)  # Reprodutibilidade
random.shuffle(lotes_dataset)

# Calcular as quantidades de cada conjunto
total = len(lotes_dataset)

n_treino = int(total * 0.70)
n_val = int(total * 0.15)

# Dividir os pares
train = lotes_dataset[:n_treino]
val = lotes_dataset[n_treino:n_treino + n_val]
test = lotes_dataset[n_treino + n_val:]

# Criar a estrutura de diretórios
base_dir = Path('Conjunto_De_Dados/INMET')

for conjunto in ['train', 'val', 'test']:
    (base_dir / conjunto / 'input').mkdir(
        parents=True, exist_ok=True
    )
    (base_dir / conjunto / 'label').mkdir(
        parents=True, exist_ok=True
    )

# Salvar os arquivos de cada conjunto
conjuntos = {
    'train': train,
    'val': val,
    'test': test
}

for nome_conjunto, pares in conjuntos.items():

    for i, par in enumerate(pares):
        # Salvar a entrada
        caminho_input = (
            base_dir / nome_conjunto / 'input' / f'{i:05d}.json'
        )

        with open(caminho_input, 'w', encoding='utf-8') as f:
            json.dump(
                par['entrada'],
                f,
                ensure_ascii=False,
                indent=4
            )

        # Salvar o label correspondente
        caminho_label = (
            base_dir / nome_conjunto / 'label' / f'{i:05d}.json'
        )

        with open(caminho_label, 'w', encoding='utf-8') as f:
            json.dump(
                par['label'],
                f,
                ensure_ascii=False,
                indent=4
            )

# Mostrar o resultado
print(f'Total de pares: {total}')
print(f'Treino: {len(train)} ({len(train)/total:.1%})')
print(f'Validação: {len(val)} ({len(val)/total:.1%})')
print(f'Teste: {len(test)} ({len(test)/total:.1%})')
print(f'Dataset salvo em: {base_dir.resolve()}')


Total de pares: 180
Treino: 125 (69.4%)
Validação: 27 (15.0%)
Teste: 28 (15.6%)
Dataset salvo em: C:\Users\caiof\Desktop\Repositórios\Nova pasta\datasets\Conjunto_De_Dados\INMET
